# WEEK 3 — DATA PIPELINE & RETRIEVAL ARCHITECTURE

Goal:
Build a structured, reusable and metadata-aware
retrieval pipeline for the proposed RAG algorithm.

In [1]:
from google.colab import files

uploaded = files.upload()

path = list(uploaded.keys())[0]

Saving cleaned_rag_document.txt to cleaned_rag_document.txt


# **Extract Text**

In [2]:
import re

def extract_text(path):
  with open(path, 'r', encoding = 'utf-8') as f:
    raw_text = f.read()

  return raw_text

text = extract_text(path)

In [3]:
len(text)

11754

# **Create Chunks**

In [4]:
def create_chunks(text, chunk_size = 500, overlap = 100):
  chunks = []
  start = 0

  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])

    start += chunk_size - overlap
  return chunks

In [5]:
chunks = create_chunks(text)
print('No of chunks:', len(chunks))

No of chunks: 30


In [6]:
chunks[1]

'text to the language model.\n\nA typical RAG pipeline contains several stages. First, documents are collected and divided into smaller chunks. Each chunk is converted into a numerical vector representation called an embedding. These embeddings are stored in a vector database. When a user submits a question, the question is also converted into an embedding. The system compares the query embedding with document embeddings and retrieves the most relevant chunks. The retrieved chunks are then provided'

# **Creating Chunks Metadata**

In [7]:
sections = [
    "Retrieval-Augmented Generation",
    "Document Chunking",
    "Text Embeddings",
    "Vector Similarity",
    "Corrective RAG",
    "Self-RAG",
    "GraphRAG",
    "ReAct",
    "Hallucination",
    "Natural Language Inference",
    "Trust Score"
]

In [8]:
def create_chunk_metadata(chunks):

    metadata_chunks = []
    current_section = "Unknown"

    for i, chunk in enumerate(chunks):

        for section in sections:

            if section.lower() in chunk.lower():
                current_section = section
                break

        metadata_chunks.append({
            "chunk_id": i,
            "section": current_section,
            "source": path,
            "chunk_text": chunk
        })

    return metadata_chunks

In [9]:
chunks_metadata = create_chunk_metadata(chunks)

print("Number of chunks:", len(chunks_metadata))
if chunks_metadata: # Check if the list is not empty
    print(chunks_metadata[0])
else:
    print("chunks_metadata is empty. Ensure 'chunks' was populated correctly before calling create_chunk_metadata.")

Number of chunks: 30
{'chunk_id': 0, 'section': 'Retrieval-Augmented Generation', 'source': 'cleaned_rag_document.txt', 'chunk_text': 'RAG Knowledge Base\n\n1. Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.\n\nA typical RAG pipeline contains several stages. First, documents are co'}


In [10]:
ids = [chunk["chunk_id"] for chunk in chunks_metadata]

print("Unique IDs:", len(set(ids)))
print("Total IDs:", len(ids))

Unique IDs: 30
Total IDs: 30


In [11]:
print(chunks_metadata[0])
print(chunks_metadata[10])
print(chunks_metadata[20])
print(chunks_metadata[-1])

{'chunk_id': 0, 'section': 'Retrieval-Augmented Generation', 'source': 'cleaned_rag_document.txt', 'chunk_text': 'RAG Knowledge Base\n\n1. Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.\n\nA typical RAG pipeline contains several stages. First, documents are co'}
{'chunk_id': 10, 'section': 'Retrieval-Augmented Generation', 'source': 'cleaned_rag_document.txt', 'chunk_text': 'the quality of retrieved information before generating the final answer.\n\nIn a basic RAG system, retrieved documents are directly passed to the language model. In CRAG, the retrieved information is evaluated. If the retrieved information is considered insuffic

# **MPNet Embeddings**

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
def generate_embedding(text):
    return model.encode(text, convert_to_numpy=True)

In [14]:
embeddings = generate_embedding(chunks_metadata[0]["chunk_text"])
print(embeddings[0])

0.039488915


# **FAISS embeddings**

In [15]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 98.9 MB/s eta 0:00:00


In [16]:
import faiss
import numpy as np

def create_faiss_index(embeddings):
  index = faiss.IndexFlatL2(embeddings.shape[1])
  index.add(embeddings)
  return index

In [17]:
texts = [chunk["chunk_text"] for chunk in chunks_metadata]

embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

In [18]:
index = create_faiss_index(embeddings)

print("Number of vectors:", index.ntotal)
print("Embedding dimension:", embeddings.shape[1])

Number of vectors: 30
Embedding dimension: 768


# Metadata-Aware **Retrieval**

In [19]:
def retrieve(query, index, chunks_metadata, model, k=3):

    query_embedding = generate_embedding(query)

    distances, indices = index.search(
        np.array([query_embedding]),
        k
    )

    results = []

    for rank, (distance, index_id) in enumerate(
        zip(distances[0], indices[0]),
        start=1
    ):

        chunk = chunks_metadata[index_id]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "section": chunk["section"],
            "distance": float(distance),
            "text": chunk["chunk_text"]
        })

    return results

In [20]:
results = retrieve(
    "What are text RAG?",
    index,
    chunks_metadata,
    model,
    k=3
)

for result in results:
    print("Rank:", result["rank"])
    print("Chunk ID:", result["chunk_id"])
    print("Section:", result["section"])
    print("Distance:", result["distance"])
    print("Text:", result["text"][:300])
    print("-" * 60)

Rank: 1
Chunk ID: 3
Section: Document Chunking
Distance: 0.8681392669677734
Text: ncorrect answer based on the retrieved information.

The quality of a RAG system depends on several factors, including document preprocessing, chunk size, chunk overlap, embedding quality, retrieval strategy, number of retrieved documents, context construction, and language model capability.

2. Doc
------------------------------------------------------------
Rank: 2
Chunk ID: 0
Section: Retrieval-Augmented Generation
Distance: 0.9088635444641113
Text: RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system
------------------------------------------------------------
Rank: 3
Chunk ID: 1
Section: Retrieval-Augmented Generation
Distance: 0.9470998048782349
Text: text

# **Hit@K Evaluation**

In [21]:
def calculate_hit_at_k(results, expected_section, k):

    top_k_results = results[:k]

    retrieved_sections = [
        result["section"].lower()
        for result in top_k_results
    ]

    return expected_section.lower() in retrieved_sections

In [22]:
query = "What is Retrieval-Augmented Generation?"
expected_section = "Retrieval-Augmented Generation"

results = retrieve(
    query,
    index,
    chunks_metadata,
    model,
    k=5
)

hit1 = calculate_hit_at_k(results, expected_section, 1)
hit3 = calculate_hit_at_k(results, expected_section, 3)
hit5 = calculate_hit_at_k(results, expected_section, 5)

print("Hit@1:", hit1)
print("Hit@3:", hit3)
print("Hit@5:", hit5)

Hit@1: True
Hit@3: True
Hit@5: True


In [23]:
!pip install -q python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.7 MB/s eta 0:00:00


In [24]:
from google.colab import files
from docx import Document

uploaded = files.upload()

question_path = list(uploaded.keys())[0]

print("Uploaded:", question_path)

Saving phase 1 questions.docx to phase 1 questions.docx
Uploaded: phase 1 questions.docx


In [25]:
doc = Document(question_path)

questions = []

for table in doc.tables:

    for row in table.rows:

        cells = [
            cell.text.strip()
            for cell in row.cells
        ]

        if len(cells) >= 3:

            question_id = cells[0]
            question = cells[1].replace("\n", " ").strip()
            expected_section = cells[2].replace("\n", " ").strip()

            if question_id in [
                f"Q{i:02d}" for i in range(1, 31)
            ]:

                questions.append({
                    "id": question_id,
                    "question": question,
                    "expected_section": expected_section
                })

print("Number of questions:", len(questions))

Number of questions: 30


In [26]:
for q in questions[:5]:
    print(q)

{'id': 'Q01', 'question': 'What is Retrieval-Augmented Generation?', 'expected_section': 'RAG'}
{'id': 'Q02', 'question': 'How does RAG use an external knowledge base?', 'expected_section': 'RAG'}
{'id': 'Q03', 'question': 'What are the main stages of a typical RAG pipeline?', 'expected_section': 'RAG'}
{'id': 'Q04', 'question': 'Why can RAG reduce hallucinations?', 'expected_section': 'RAG'}
{'id': 'Q05', 'question': 'Does RAG guarantee that generated answers are correct?', 'expected_section': 'RAG'}


In [27]:
def calculate_hit_at_k(results, expected_section, k):

    top_k_results = results[:k]

    retrieved_sections = [
        result["section"].lower()
        for result in top_k_results
    ]

    return expected_section.lower() in retrieved_sections

In [28]:
import pandas as pd

evaluation_results = []

for q in questions:

    query = q["question"]
    expected_section = q["expected_section"]

    results = retrieve(
        query,
        index,
        chunks_metadata,
        model,
        k=5
    )

    hit1 = calculate_hit_at_k(
        results,
        expected_section,
        1
    )

    hit3 = calculate_hit_at_k(
        results,
        expected_section,
        3
    )

    hit5 = calculate_hit_at_k(
        results,
        expected_section,
        5
    )

    evaluation_results.append({

        "question_id": q["id"],

        "question": query,

        "expected_section": expected_section,

        "retrieved_chunk_ids": [
            r["chunk_id"]
            for r in results
        ],

        "retrieved_sections": [
            r["section"]
            for r in results
        ],

        "distances": [
            r["distance"]
            for r in results
        ],

        "hit@1": hit1,

        "hit@3": hit3,

        "hit@5": hit5
    })

evaluation_df = pd.DataFrame(
    evaluation_results
)

print(
    "Evaluation completed:",
    len(evaluation_df),
    "questions"
)

Evaluation completed: 30 questions


In [29]:
evaluation_df[
    [
        "question_id",
        "expected_section",
        "retrieved_sections",
        "hit@1",
        "hit@3",
        "hit@5"
    ]
].head(10)

,question_id,expected_section,retrieved_sections,hit@1,hit@3,hit@5
0,Q01,RAG,"[Retrieval-Augmented Generation, Retrieval-Aug...",False,False,False
1,Q02,RAG,"[Retrieval-Augmented Generation, Self-RAG, Sel...",False,False,False
2,Q03,RAG,"[Retrieval-Augmented Generation, Document Chun...",False,False,False
3,Q04,RAG,"[Hallucination, Hallucination, Hallucination, ...",False,False,False
4,Q05,RAG,"[Trust Score, Retrieval-Augmented Generation, ...",False,False,False
5,Q06,RAG,"[Document Chunking, Self-RAG, Retrieval-Augmen...",False,False,False
6,Q07,Document Chunking,"[Document Chunking, Document Chunking, Text Em...",True,True,True
7,Q08,Document Chunking,"[Document Chunking, Text Embeddings, Document ...",True,True,True
8,Q09,Document Chunking,"[Document Chunking, Document Chunking, Text Em...",True,True,True
9,Q10,Document Chunking,"[Text Embeddings, Document Chunking, Document ...",False,True,True


In [30]:
hit1_score = evaluation_df["hit@1"].mean() * 100
hit3_score = evaluation_df["hit@3"].mean() * 100
hit5_score = evaluation_df["hit@5"].mean() * 100

print("================================")
print("       WEEK 3 HIT@K RESULTS")
print("================================")

print(f"Total Questions: {len(evaluation_df)}")

print(f"Hit@1: {hit1_score:.2f}%")
print(f"Hit@3: {hit3_score:.2f}%")
print(f"Hit@5: {hit5_score:.2f}%")

       WEEK 3 HIT@K RESULTS
Total Questions: 30
Hit@1: 50.00%
Hit@3: 60.00%
Hit@5: 60.00%


In [31]:
evaluation_df.to_csv(
    "week3_hitk_evaluation.csv",
    index=False
)

print("Saved: week3_hitk_evaluation.csv")

Saved: week3_hitk_evaluation.csv
